# Audit dữ liệu backtest
Grain: ticker × rebalance_date. Chỉ đánh giá input signal; không có performance/backtest results.
Xem `backtest-data-quality.md` cho policy, limits và các thay đổi source.


In [ ]:
from pathlib import Path
import sys, json, csv
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
sys.path.insert(0, str(root))
from research.backtest_quality import audit


## Chạy lại read-only audit trên warehouse hiện tại
Không chạy cùng lúc Airflow/dbt đang ghi warehouse.


In [ ]:
assessment = audit(root/'data/warehouse.duckdb', root/'data/reports/backtest_quality')
assessment['latest']


## Truy vấn và evidence snapshot trước/sau
Snapshot aggregate được capture trong remediation 15/10/2026; không tự coi là baseline của các lần build tương lai.


In [ ]:
sql = (root/'research/sql/backtest_inputs.sql').read_text(encoding='utf-8')
print(sql)
with (root/'research/evidence/coverage_before_after_2026-10-15.csv').open(encoding='utf-8') as f:
    before_after = list(csv.DictReader(f))
before_after[-1]


In [ ]:
with (root/'data/reports/backtest_quality/ticker_diagnostics.csv').open(encoding='utf-8') as f:
    diagnostics = list(csv.DictReader(f))
latest = [r for r in diagnostics if r['rebalance_date']==assessment['latest']['rebalance_date'].isoformat()]
[r for r in latest if r['is_score_eligible']=='True' and r['is_research_signal_eligible']!='True']


## Điều kiện diễn giải
- Membership monthly observations; sector current metadata.
- 550/400/200 ngày là thresholds policy, không phải kết quả tối ưu performance.
- Không dùng future-return availability để lọc mẫu.
- IFRS/tax-inclusive revenue/broader Capex/allocated SBC không được alias ngầm.
